TAMER 베이스라인 모델 불러오기

In [1]:
# Connect Google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# TAMER GitHub repo clone (최초 1회)
%cd "/content/drive/MyDrive/AI_Project"
# !git clone https://github.com/qingzhenduyu/TAMER.git
%cd TAMER

/content/drive/MyDrive/AI_Project
/content/drive/MyDrive/AI_Project/TAMER


In [3]:
# Install requirements
!pip install einops==0.3.0
!pip install editdistance
!pip install pytorch-lightning==1.9.4

# for pytorch-lightning CLI
!pip install jsonargparse[signatures]==3.17.0

# dev-dependency
!pip install flake8==3.9.0
!pip install black==22.3.0
!pip install isort==5.8.0
!pip install jupyter==1.0.0
!pip install opencv-python-headless --no-cache-dir # -headless option
!pip install matplotlib==3.5.1

# for test in crohme
!pip install typer==0.4.1
!pip install beautifulsoup4==4.10.0
!pip install lxml>=4.9.1

In [4]:
# Run test
import os
import torch
import pytorch_lightning as pl
from tamer.datamodule.datamodule import HMEDatamodule
from tamer.lit_tamer import LitTAMER

# Allow unpickle
from torch.serialization import add_safe_globals
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint
add_safe_globals([ModelCheckpoint])

def test_model(dataset_name, checkpoint_path, data_root, eval_batch_size=1, num_workers=2):
    # Load dataset
    dm = HMEDatamodule(
        folder=data_root,
        test_folder=dataset_name,
        eval_batch_size=eval_batch_size,
        num_workers=num_workers
    )
    dm.setup("test")

    # Load model
    checkpoint = torch.load(checkpoint_path, map_location="cuda" if torch.cuda.is_available() else "cpu")
    model = LitTAMER.load_from_checkpoint(checkpoint_path, map_location="cuda" if torch.cuda.is_available() else "cpu")

    # Test
    print(f"\nTesting on {data_root}/{dataset_name} dataset...")
    trainer = pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        logger=False,
        enable_progress_bar=True
    )
    trainer.test(model, datamodule=dm)
    print("-" * 50)

변환한 im2latex 작동하는지 테스트

In [5]:
# # im2latex_formulas.lst를 dictionary.txt로 변환 (최초 1회)
# def create_dictionary(formula_file, save_path="/content/drive/MyDrive/AI_Project/im2latex-100k/dictionary.txt"):
#     tokens = set()
#     with open(formula_file, 'r', encoding='latin1') as f:
#         for line in f:
#             expr = line.strip()
#             tokens.update(expr.split())  # 공백 기준 토큰 분리

#     sorted_tokens = sorted(tokens)
#     with open(save_path, 'w', encoding='utf-8') as f:
#         for token in sorted_tokens:
#             f.write(token + '\n')

#     print(f"Saved {len(sorted_tokens)} tokens to {save_path}")

In [6]:
# create_dictionary("/content/drive/MyDrive/AI_Project/im2latex-100k/im2latex_formulas.lst")

In [7]:
# # Check whether caption.txt. and images.pkl matched in im2latex
# import os
# import pickle

# base_path = '/content/drive/MyDrive/AI_Project/im2latex-100k/test'
# print(f"Checking dataset for test")

# image_pkl_path = os.path.join(base_path, 'images.pkl')
# caption_txt_path = os.path.join(base_path, 'caption.txt')

# # Check if files exist
# if not os.path.exists(image_pkl_path):
#     print(f"Missing file: {image_pkl_path}")
# if not os.path.exists(caption_txt_path):
#     print(f"Missing file: {caption_txt_path}")

# # Load data
# with open(image_pkl_path, 'rb') as f:
#     images = pickle.load(f)
# with open(caption_txt_path, 'r') as f:
#     captions = f.readlines()

# # Check for missing image keys
# missing_images = []
# for line in captions:
#     img_name = line.strip().split()[0]
#     if img_name not in images:
#         missing_images.append(img_name)

# if missing_images:
#     print(f"{len(missing_images)} in caption.txt missed")
#     print("Example missing keys:", missing_images[:5], "\n")
# else:
#     print(f"All {len(captions)} in caption.txt exist\n")

In [8]:
# # 데이터셋 클래스 import
# from tamer.datamodule.datamodule import HMEDatamodule  # 예시, 실제 데이터셋 클래스명 확인 필요
# import numpy as np
# import torch

# # 데이터셋 초기화 (경로, 설정 등 확인 필요)
# dm = HMEDatamodule(
#     folder="/content/drive/MyDrive/AI_Project/im2latex-100k",
#     test_folder="test",
#     eval_batch_size=1,
#     num_workers=0
# )
# dm.setup("test")

# # 임의 샘플 확인
# idx = 0
# test_dataset = dm.test_dataloader().dataset
# sample = test_dataset[0]

# print(f"Sample at idx={idx}:")
# print(f"Type of sample: {type(sample)}")

# # 샘플이 보통 (image, caption) 튜플이라면
# if isinstance(sample, tuple) and len(sample) == 2:
#     img, caption = sample
#     print(f"Image type: {type(img)}")
#     if isinstance(img, np.ndarray):
#         print(f"Image shape: {img.shape}, dtype: {img.dtype}")
#     elif isinstance(img, torch.Tensor):
#         print(f"Image shape: {img.shape}, dtype: {img.dtype}")
#     else:
#         print("Image is neither numpy.ndarray nor torch.Tensor")
#     print(f"Caption type: {type(caption)}")
#     print(f"Caption sample: {caption[:50]}")  # 앞부분만 출력
# else:
#     print("Sample format unexpected, expected tuple (image, caption)")
#     print(f"Sample contents: {sample}")
#     print(f"Length of sample: {len(sample)}")


In [9]:
# # 예시 (dataset.pkl 파일을 불러온 후)
# with open('/content/drive/MyDrive/AI_Project/TAMER/data/hme100k/test/images.pkl', 'rb') as f:
#     data = pickle.load(f)

# print(len(data))           # 전체 샘플 수
# print(type(data))       # <class 'tuple'>
# print(data.keys())        # 2 또는 3

In [10]:
# # 예시 (dataset.pkl 파일을 불러온 후)
# with open('/content/drive/MyDrive/AI_Project/im2latex-100k/test/images.pkl', 'rb') as f:
#     data_100 = pickle.load(f)

# print(len(data_100))           # 전체 샘플 수
# print(type(data_100))       # <class 'tuple'>
# print(data_100.keys())        # 2 또는 3

In [11]:
# # HME100K 체크포인트 버전
# ckpt_path = "/content/drive/MyDrive/AI_Project/TAMER/lightning_logs/version_1/checkpoints/epoch=51-step=162967-val_ExpRate=0.6851.ckpt"

# test_model(dataset_name="test", checkpoint_path=ckpt_path, data_root="/content/drive/MyDrive/AI_Project/im2latex-100k", eval_batch_size=1, num_workers=0)

변환한 CROPME 작동하는지 테스트

In [12]:
# Check whether caption.txt. and images.pkl matched in CROHME
import os
import pickle

# List of dataset folders to check
year = '2014'
base_path = './data/cropme'

print(f"Checking dataset for year: {year}")

image_pkl_path = os.path.join(base_path, year, 'images.pkl')
caption_txt_path = os.path.join(base_path, year, 'caption.txt')

# Check if files exist
if not os.path.exists(image_pkl_path):
    print(f"Missing file: {image_pkl_path}")
if not os.path.exists(caption_txt_path):
    print(f"Missing file: {caption_txt_path}")

# Load data
with open(image_pkl_path, 'rb') as f:
    images = pickle.load(f)
with open(caption_txt_path, 'r') as f:
    captions = f.readlines()

# Check for missing image keys
missing_images = []
for line in captions:
    img_name = line.strip().split()[0]
    if img_name not in images:
        missing_images.append(img_name)

if missing_images:
    print(f"{len(missing_images)} in caption.txt missed")
    print("Example missing keys:", missing_images[:5], "\n")
else:
    print(f"All {len(captions)} in caption.txt exist\n")

Checking dataset for year: 2014
All 986 in caption.txt exist



In [13]:
# Test CROHME 2014, 2016, 2019
ckpt_path = "/content/drive/MyDrive/AI_Project/TAMER/lightning_logs/version_0/checkpoints/epoch=315-step=118815-val_ExpRate=0.6113.ckpt"
year = "2014"
test_model(dataset_name=year, checkpoint_path=ckpt_path, data_root="data/cropme", eval_batch_size=4, num_workers=2)

Load data from: data/cropme
Extract data from: 2014, with data size: 986
total  247 batch data loaded


/usr/local/lib/python3.11/dist-packages/pytorch_lightning/utilities/migration/migration.py:195: PossibleUserWarning: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
  rank_zero_warn(
INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.4.9 to v1.9.4. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file lightning_logs/version_0/checkpoints/epoch=315-step=118815-val_ExpRate=0.6113.ckpt`
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first w


Testing on data/cropme/2014 dataset...
Extract data from: 2014, with data size: 986
total  247 batch data loaded


Testing: 0it [00:00, ?it/s]

Validation ExpRate: 0.0
Validation 1-error Rate: 0.000000000000
Validation 2-error Rate: 0.000000000000
--------------------------------------------------
